In [1]:
from pathlib import Path
from scipy import sparse

import pandas as pd
import numpy as np
import xarray as xr
import glob
import os


In [3]:
terr = xr.open_dataset("/lfs/home/lama/src/corrdiff_input/ref_grid/wrf_208x208_grid_coords.nc")

In [4]:
lat_min, lat_max = 20.0, 29.0
lon_min, lon_max = 117.0, 126.0

with xr.open_dataset(files_list[0]) as ref:
    mask = (
            (ref["XLAT"].isel(Time=0)>=lat_min) & (ref["XLAT"].isel(Time=0)<=lat_max) &
            (ref["XLONG"].isel(Time=0)>=lon_min) & (ref["XLONG"].isel(Time=0)<=lon_max)) # mask is of xarray.DataArray

    lat=ref["XLAT"].isel(Time=0).values.ravel()[mask.values.ravel()]
    lon=ref["XLONG"].isel(Time=0).values.ravel()[mask.values.ravel()]



pts_lat = xr.DataArray(lat, dims="points")
pts_lon = xr.DataArray(lon, dims="points") 




regrid = [] # regrid[0], ..., regrid[196]
for track in all_track_data:

    track_times = pd.to_datetime(track["time"].values)

    wind_anom_tw_time_slice = wind_anom_tw.sel(time=track_times)

    # max anomaly during track lifetime
    wind_anom_tw_max = wind_anom_tw_time_slice.max(dim="time")

    # sort before interpolation
    wind_anom_tw_max = wind_anom_tw_max.sortby(["latitude", "longitude"])

    # interpolate ERA5 anomaly field to WRF mask points
    r = wind_anom_tw_max.interp(latitude=pts_lat,
                                longitude=pts_lon,
                                method="linear")

    regrid.append(r)

<xarray.Dataset> Size: 1MB
Dimensions:   (south_north: 208, west_east: 208)
Coordinates:
    XLAT      (south_north, west_east) float32 173kB ...
    XLONG     (south_north, west_east) float32 173kB ...
Dimensions without coordinates: south_north, west_east
Data variables:
    TER       (south_north, west_east) float32 173kB ...
    LANDMASK  (south_north, west_east) float32 173kB ...
    SLOPE     (south_north, west_east) float32 173kB ...
    ASPECT    (south_north, west_east) float32 173kB ...
Attributes:
    description:  New CorrDiff Training REF grid 208x208